# 04 — NumPy for AI Engineers

    - Manipulate arrays and shapes
- Explain broadcasting and axes
- Implement vectorized ML calculations

    **Estimated study/practice time:** 3–4 hours  
    **Prerequisites:** Python data structures; basic algebra

    **Priority legend:** 🔴 **Must Know** · 🟠 **Important** · 🟢 **Good to Know**

    ## Table of contents

    1. [Arrays and shapes](#arrays-and-shapes)
2. [Indexing and masking](#indexing-and-masking)
3. [Vectorization and broadcasting](#vectorization-and-broadcasting)
4. [Reshape and axes](#reshape-and-axes)
5. [Linear algebra](#linear-algebra)
6. [20 exercises](#20-exercises)

    > Active-practice rule: run examples, predict output before revealing it, complete every `TODO`, then compare with the solution below it.


## Arrays and shapes

🔴 **Must Know** — **Concept:** `ndarray` is a homogeneous n-dimensional numeric container. **Why it matters:** embeddings, batches, weights, and predictions depend on shape-correct vectorized computation. **Syntax:** `np.array`, `shape`, `ndim`, `size`, and `dtype`. **How NumPy executes it:** homogeneous buffers let compiled loops operate without per-element Python dispatch. **Complexity:** construction/copying is O(number of elements); metadata access is O(1). **Common mistake:** confusing `(n,)` with `(n,1)`. **Interview question:** list versus ndarray? Flexible reference collection versus homogeneous numerical tensor. **Exercise/solution:** begin with *vector norm* and *flatten images*.

```text
[[1, 2, 3],     2 rows
 [4, 5, 6]]  ×  3 columns  → shape (2, 3), ndim 2, size 6
```


In [ ]:
import numpy as np
rng = np.random.default_rng(42)
np.random.seed(42)
x = np.array([[1, 2, 3], [4, 5, 6]])
print(x.shape, x.ndim, x.size, x.dtype)
print(np.zeros((2, 3)), np.ones(3), np.arange(0, 6, 2), np.linspace(0, 1, 5))
normal_samples = rng.standard_normal((2, 3))  # reproducible replacement for legacy randn
legacy_randn_example = np.random.randn(2, 3)  # explicitly requested legacy API
print(normal_samples, legacy_randn_example)


## Indexing and masking

🔴 **Must Know** — **Concept:** indices and slices select positions; boolean masks select by condition. **Why it matters:** ML preprocessing often filters predictions or chooses feature columns. **Syntax:** `x[row,column]`, `x[:,0]`, `x[0,:]`, `x[x>0]`. **How NumPy executes it:** basic slices are usually views; advanced/boolean indexing usually copies. **Complexity:** scalar indexing O(1); selecting k outputs O(k). **Common mistake:** mutating a slice view unintentionally. **Interview question:** what does `x[:,0]` mean? Every row, first column. **Exercise/solution:** complete *filter predictions* and *top index per row*.


In [ ]:
x = np.array([[1, 2, 3], [4, 5, 6]])
print("first", x[0, 0], "column", x[:, 0], "row", x[0, :], "block", x[:2, 1:])
scores = np.array([.2, .8, .95])
print(scores[scores >= .5])


## Vectorization and broadcasting

🔴 **Must Know** — **Concept:** vectorization applies operations to whole arrays; broadcasting expands compatible size-1 dimensions conceptually. **Why it matters:** readable, efficient feature and batch operations avoid Python loops. **Syntax:** `arr*2`, `matrix+vector`. **How NumPy executes it:** compiled strided loops align shapes from the right without necessarily copying broadcast values. **Complexity:** O(number of result elements), with result-sized space. **Common mistake:** comparing only total element counts. **Interview question:** when are aligned dimensions compatible? Equal or one is 1. **Exercise/solution:** complete *standardize columns*, *center rows*, and *linear layer*.

```text
matrix (3, 4)
      +    vector (4,)
      =    result (3, 4)

(3, 4) + (3,)  ✗ trailing dimensions 4 and 3 conflict
```


In [ ]:
matrix = np.arange(12).reshape(3, 4)
feature_bias = np.array([10, 20, 30, 40])
print(matrix + feature_bias)
print(matrix * 2)  # vectorized scalar broadcast


**Common interview mistake:** guessing broadcast compatibility from total element count. Compare aligned dimensions from right to left.

## Reshape and axes

🔴 **Must Know** — **Concept:** reshape changes dimensional interpretation while axes name dimensions reduced by aggregations. **Why it matters:** batch/image layouts and row/column statistics must preserve intended samples and features. **Syntax:** `reshape`, `ravel`, `flatten`, and `sum(axis=...)`. **How NumPy executes it:** reshape/ravel return views when strides permit; flatten copies; a reduction removes the selected axis. **Complexity:** view reshape O(1), flatten O(n), reductions O(n). **Common mistake:** saying axis=0 means “rows” without explaining that rows are collapsed. **Interview question:** axis=0 output? One aggregate per column. **Exercise/solution:** complete *column means* and *row max*.

```text
  1 2 3
  4 5 6
axis=0 ↓ collapse rows → [5, 7, 9]  (one result per column)
axis=1 → collapse cols → [6, 15]    (one result per row)
```


In [ ]:
x = np.array([[1,2,3],[4,5,6]])
print(x.reshape(3, 2), x.ravel(), x.flatten())
print("sum0", x.sum(axis=0), "sum1", x.sum(axis=1))
print(np.mean(x), np.std(x), np.min(x), np.max(x), np.argmax(x, axis=1))


## Linear algebra

🔴 **Must Know**

**Concept.** Element-wise multiplication and matrix multiplication are distinct operations.  
**Why it matters.** Neural layers use `X @ W + b`; embedding similarity uses dot products and norms.  
**Syntax.** `A * B`, `A @ B`, `np.dot(a,b)`.  
**How NumPy executes it.** Element-wise shapes broadcast; matrix multiplication contracts matching inner dimensions.  
**Complexity.** Dense `(b,f) @ (f,o)` is O(bfo), output space O(bo).  
**Common mistake.** Using `*` where `@` is required.  
**Interview question.** What is the output of `(b,f) @ (f,o)`? `(b,o)`.  
**Exercise/solution.** Complete *pairwise scores* and *cosine similarity*.

```text
A * B → element-wise; shapes must broadcast
A @ B → matrix multiplication; inner dimensions must match
X (batch, features) @ W (features, outputs) + b (outputs,)
  → predictions (batch, outputs)
```

Vector dot products measure weighted alignment and form the numerator of cosine similarity.


In [ ]:
A = np.array([[1,2],[3,4]])
B = np.array([[2,0],[1,2]])
print("elementwise", A * B)
print("matrix", A @ B)
a, b = np.array([1., 2.]), np.array([3., 4.])
cosine = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print("dot", np.dot(a, b), "cosine", cosine)


## 20 exercises


### Exercise — 🟢 Easy: 1. vector_norm

        **Goal:** Return L2 norm.

        **Example**

        ```text
        Input: [3.0, 4.0]
Output: 5.0
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def vector_norm(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def vector_norm(x):
    return np.linalg.norm(x)

assert np.isclose(vector_norm(np.array([3.,4.])),5)


**Complexity**

```text
Time: O(n); Space: O(1)
```

**Common mistake:** Forgetting the square root by returning the sum of squares.


### Exercise — 🟢 Easy: 2. normalize_vector

        **Goal:** Return a unit vector; leave zero vector unchanged.

        **Example**

        ```text
        Input: [3.0, 4.0]
Output: [0.6, 0.8]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def normalize_vector(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def normalize_vector(x):
    return x / norm if (norm := np.linalg.norm(x)) else x.copy()

assert np.allclose(normalize_vector(np.array([3.,4.])),[.6,.8])


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Dividing a zero vector by zero.


### Exercise — 🟢 Easy: 3. standardize_columns

        **Goal:** Z-score each column.

        **Example**

        ```text
        Input: [[1, 2], [3, 4]]
Output: each output column has mean 0 and standard deviation 1
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def standardize_columns(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def standardize_columns(x):
    return (x - x.mean(axis=0)) / x.std(axis=0)

assert np.allclose(standardize_columns(np.array([[1.,2.],[3.,4.]])).mean(0),0)


**Complexity**

```text
Time: O(rc); Space: O(rc)
```

**Common mistake:** Using a global mean/std instead of axis=0.


### Exercise — 🟢 Easy: 4. row_max

        **Goal:** Return maximum per row.

        **Example**

        ```text
        Input: [[1, 3], [4, 2]]
Output: [3, 4]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def row_max(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def row_max(x):
    return x.max(axis=1)

assert np.array_equal(row_max(np.array([[1,3],[4,2]])),[3,4])


**Complexity**

```text
Time: O(rc); Space: O(r)
```

**Common mistake:** Reducing axis=0 and returning column maxima.


### Exercise — 🟢 Easy: 5. cosine_similarity

        **Goal:** Compute cosine similarity, rejecting zero norms.

        **Example**

        ```text
        Input: [1, 0], [1, 0]
Output: 1.0
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def cosine_similarity(a, b):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def cosine_similarity(a, b):
    return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)) if np.linalg.norm(a)*np.linalg.norm(b) else 0.0

assert np.isclose(cosine_similarity(np.array([1.,0.]),np.array([1.,0.])),1)


**Complexity**

```text
Time: O(n); Space: O(1)
```

**Common mistake:** Omitting either vector norm from the denominator.


### Exercise — 🟢 Easy: 6. flatten_images

        **Goal:** Flatten `(batch,height,width)` to `(batch,-1)`.

        **Example**

        ```text
        Input shape: (2, 3, 4)
Output shape: (2, 12)
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def flatten_images(images):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def flatten_images(images):
    return images.reshape(images.shape[0], -1)

assert flatten_images(np.zeros((2,3,4))).shape==(2,12)


**Complexity**

```text
Time: O(1) view; Space: O(1) when possible
```

**Common mistake:** Flattening the batch dimension together with pixels.


### Exercise — 🟢 Easy: 7. filter_predictions

        **Goal:** Keep scores at least threshold.

        **Example**

        ```text
        Input: [0.2, 0.8], threshold=0.5
Output: [0.8]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def filter_predictions(scores, threshold):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def filter_predictions(scores, threshold):
    return scores[scores >= threshold]

assert np.array_equal(filter_predictions(np.array([.2,.8]),.5),[.8])


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Using `>` when the threshold is inclusive.


### Exercise — 🟢 Easy: 8. linear_layer

        **Goal:** Compute X @ W + b.

        **Example**

        ```text
        Input shapes: X=(2,3), W=(3,1), b=(1,)
Output shape: (2,1)
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def linear_layer(X, W, b):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def linear_layer(X, W, b):
    return X @ W + b

assert linear_layer(np.ones((2,3)),np.ones((3,1)),np.array([1])).shape==(2,1)


**Complexity**

```text
Time: O(bfo); Space: O(bo)
```

**Common mistake:** Using element-wise `*` instead of matrix `@`.


### Exercise — 🟠 Medium: 9. one_hot

        **Goal:** One-hot encode integer labels.

        **Example**

        ```text
        Input: labels=[0,2], classes=3
Output: [[1,0,0],[0,0,1]]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def one_hot(labels, num_classes):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def one_hot(labels, num_classes):
    return np.eye(num_classes, dtype=int)[labels]

assert np.array_equal(one_hot(np.array([0,2]),3),[[1,0,0],[0,0,1]])


**Complexity**

        ```text
        Time: O(nc)
Space: O(nc)
        ```

        **Common mistake:** Using a class count smaller than the largest label plus one.


### Exercise — 🟠 Medium: 10. column_means

        **Goal:** Return column means.

        **Example**

        ```text
        Input: [[1,2],[3,4]]
Output: [2,3]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def column_means(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def column_means(x):
    return x.mean(axis=0)

assert np.array_equal(column_means(np.array([[1,2],[3,4]])),[2,3])


**Complexity**

```text
Time: O(rc); Space: O(c)
```

**Common mistake:** Reducing axis=1.


### Exercise — 🟠 Medium: 11. row_softmax

        **Goal:** Stable softmax over each row.

        **Example**

        ```text
        Input: [[1.0, 2.0]]
Output: one row of probabilities summing to 1
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def row_softmax(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def row_softmax(x):
    shifted = x - x.max(axis=1, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=1, keepdims=True)

assert np.allclose(row_softmax(np.array([[1.,2.]]).copy()).sum(1),1)


**Complexity**

        ```text
        Time: O(rc)
Space: O(rc)
        ```

        **Common mistake:** Exponentiating unshifted large logits and overflowing.


### Exercise — 🟠 Medium: 12. clip_scores

        **Goal:** Clip values to [0,1].

        **Example**

        ```text
        Input: [-1, 0.5, 2]
Output: [0, 0.5, 1]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def clip_scores(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def clip_scores(x):
    return np.clip(x, 0, 1)

assert np.array_equal(clip_scores(np.array([-1,.5,2])),[0,.5,1])


**Complexity**

        ```text
        Time: O(n)
Space: O(n)
        ```

        **Common mistake:** Mutating the input when a new result is expected.


### Exercise — 🟠 Medium: 13. replace_nan

        **Goal:** Replace NaN with column means.

        **Example**

        ```text
        Input: [[1, NaN], [3, 2]]
Output: no NaNs; missing value uses its column mean
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def replace_nan(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def replace_nan(x):
    return np.where(np.isnan(x), np.nanmean(x,axis=0), x)

assert not np.isnan(replace_nan(np.array([[1.,np.nan],[3.,2.]]))).any()


**Complexity**

        ```text
        Time: O(rc)
Space: O(rc)
        ```

        **Common mistake:** Taking one global mean instead of a mean per column.


### Exercise — 🟠 Medium: 14. top_index_per_row

        **Goal:** Return argmax per row.

        **Example**

        ```text
        Input: [[1,2],[3,0]]
Output: [1,0]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def top_index_per_row(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def top_index_per_row(x):
    return x.argmax(axis=1)

assert np.array_equal(top_index_per_row(np.array([[1,2],[3,0]])),[1,0])


**Complexity**

```text
Time: O(rc); Space: O(r)
```

**Common mistake:** Returning maximum values rather than their indices.


### Exercise — 🟠 Medium: 15. pairwise_scores

        **Goal:** Compute X @ Y.T.

        **Example**

        ```text
        Input shapes: X=(2,3), Y=(4,3)
Output shape: (2,4)
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def pairwise_scores(X, Y):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def pairwise_scores(X, Y):
    return X @ Y.T

assert pairwise_scores(np.ones((2,3)),np.ones((4,3))).shape==(2,4)


**Complexity**

```text
Time: O(nmd); Space: O(nm)
```

**Common mistake:** Forgetting to transpose Y.


### Exercise — 🟠 Medium: 16. center_rows

        **Goal:** Subtract each row mean.

        **Example**

        ```text
        Input: [[1,3]]
Output: [[-1,1]]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def center_rows(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def center_rows(x):
    return x - x.mean(axis=1,keepdims=True)

assert np.allclose(center_rows(np.array([[1.,3.]])).mean(1),0)


**Complexity**

        ```text
        Time: O(rc)
Space: O(rc)
        ```

        **Common mistake:** Dropping `keepdims=True` and breaking broadcasting.


### Exercise — 🟠 Medium: 17. batch_indices

        **Goal:** Return shuffled indices from a seeded RNG.

        **Example**

        ```text
        Input: n=4, seed=42
Output: a reproducible permutation of [0,1,2,3]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def batch_indices(n, seed):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def batch_indices(n, seed):
    return np.random.default_rng(seed).permutation(n)

assert np.array_equal(batch_indices(4,42),batch_indices(4,42))


**Complexity**

        ```text
        Time: O(n)
Space: O(n)
        ```

        **Common mistake:** Using an unseeded global RNG and losing reproducibility.


### Exercise — 🟠 Medium: 18. accuracy

        **Goal:** Compute fraction of equal labels.

        **Example**

        ```text
        Input: true=[1,0], predicted=[1,1]
Output: 0.5
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def accuracy(y_true, y_pred):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

assert np.isclose(accuracy(np.array([1,0]),np.array([1,1])),.5)


**Complexity**

```text
Time: O(n); Space: O(n)
```

**Common mistake:** Using Python `and` instead of element-wise equality.


### Exercise — 🟠 Medium: 19. confusion_counts

        **Goal:** Build a class-by-class count matrix.

        **Example**

        ```text
        Input: true=[0,1], predicted=[0,1], classes=2
Output: [[1,0],[0,1]]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def confusion_counts(y_true, y_pred, classes):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def confusion_counts(y_true, y_pred, classes):
    counts = np.zeros((classes, classes), dtype=int)
    np.add.at(counts, (y_true, y_pred), 1)
    return counts

assert np.array_equal(confusion_counts(np.array([0,1]),np.array([0,1]),2),np.eye(2,dtype=int))


**Complexity**

```text
Time: O(n); Space: O(c²)
```

**Common mistake:** Overwriting repeated class pairs instead of accumulating them.


### Exercise — 🟠 Medium: 20. minmax_columns

        **Goal:** Scale columns to [0,1], guarding constant columns.

        **Example**

        ```text
        Input: [[1,2],[3,2]]
Output: [[0,0],[1,0]]
        ```

        <details><summary>Hint 1</summary>Identify the output shape.</details>
        <details><summary>Hint 2</summary>Use an axis-aware vectorized operation.</details>


In [ ]:
def minmax_columns(x):
    # TODO: Write your NumPy solution here
    pass


#### Solution — reveal only after attempting


In [ ]:
def minmax_columns(x):
    minimum = x.min(axis=0)
    span = x.max(axis=0) - minimum
    safe_span = np.where(span == 0, 1, span)
    return (x - minimum) / safe_span

assert np.allclose(minmax_columns(np.array([[1.,2.],[3.,2.]]))[:,0],[0,1])


**Complexity**

        ```text
        Time: O(rc)
Space: O(rc)
        ```

        **Common mistake:** Dividing constant columns by zero.


## Key takeaways

    - Shapes are part of the algorithm.
- Broadcasting aligns from the right.
- `*` and `@` mean different operations.

    ## Mistakes to avoid

    - Reducing the wrong axis.
- Dividing by zero norms.
- Accidentally mutating through a view.

    ## 10 rapid-fire interview questions

    1. **Why ndarray over a list for matrices?** <details><summary>Answer</summary>Homogeneous storage and vectorized numerical operations.</details>
2. **What does shape describe?** <details><summary>Answer</summary>Length of every array dimension.</details>
3. **What does ndim return?** <details><summary>Answer</summary>Number of dimensions.</details>
4. **What does size return?** <details><summary>Answer</summary>Total element count.</details>
5. **What does dtype describe?** <details><summary>Answer</summary>Element representation/type.</details>
6. **What does `x[:,0]` select?** <details><summary>Answer</summary>First column.</details>
7. **What does `x[0,:]` select?** <details><summary>Answer</summary>First row.</details>
8. **What is boolean masking?** <details><summary>Answer</summary>Selecting elements/rows where a boolean condition is true.</details>
9. **What is vectorization?** <details><summary>Answer</summary>Expressing whole-array operations without Python element loops.</details>
10. **What is broadcasting?** <details><summary>Answer</summary>Rules for combining compatible differently shaped arrays.</details>

    ## Readiness checklist

    - [ ] I can predict shapes.
- [ ] I can normalize rows/columns.
- [ ] I can explain X @ W + b.
